In [3]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [4]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("HARTEVATN_processed.csv")
df_weather = load_blob_csv("HARTEVATN_weather.csv")
df_norgespris = load_blob_csv("hartevatn_norgespris.csv")

In [5]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.regression import (
    prepare_norgespris_regression_data,
    fit_best_norgespris_model
)

regression_df = prepare_norgespris_regression_data(
    df_consumption,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)

results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print("Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.")
print(f"Gjennomsnittlig andel med Norgespris i etterperioden: {metrics['mean_share_post']:.2%}")
print(f"Effekt ved +10 prosentpoeng høyere andel: {metrics['effect_10pp_pct']:.2f}%")
print(f"Effekt ved 0% -> 100% andel: {metrics['effect_full_share_pct']:.2f}%")
print()
print(f"Total observert forbruk i etterperioden: {metrics['total_observed_post_kwh']:,.0f} kWh")
print(f"Estimert total effekt av Norgespris i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Dette tilsvarer: {metrics['attributable_share_of_post_kwh_pct']:.2f}% av totalforbruket i etterperioden")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag (etterperiode): {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")
print()
print(results.summary)
print()
print(f"Koeffisient for norgespris_share (log-skala): {metrics['beta_share']:.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.4g}")

Observasjoner totalt: 8,724,360
Brukt i modell: 8,724,360
R2 (within): 0.1640
Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.
Gjennomsnittlig andel med Norgespris i etterperioden: 84.44%
Effekt ved +10 prosentpoeng høyere andel: 0.32%
Effekt ved 0% -> 100% andel: 3.21%

Total observert forbruk i etterperioden: 6,445,471 kWh
Estimert total effekt av Norgespris i etterperioden: 277,891 kWh
Dette tilsvarer: 4.31% av totalforbruket i etterperioden
Per kunde i etterperioden: 127.5 kWh
Per kunde per dag (etterperiode): 1.634 kWh

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1640
Estimator:                   PanelOLS   R-squared (Between):              0.0000
No. Observations:             8724360   R-squared (Within):               0.1640
Date:                Mon, Apr 20 2026   R-squared (Overall):              0.0878
Time:              